# Jigsaw Toxic Comment Classification - Data Processing & Feature Engineering

This notebook builds a reproducible preprocessing + feature engineering pipeline. Key principles:

- Split first (to avoid leakage).
- Fit preprocessing only on train.
- Use sklearn Pipeline / ColumnTransformer.
- Output: X_train, X_test, y_train, y_test, and preprocess_pipeline.

---
# Phase 1:
- Import the needed libraries and configure them respectly.
- Load the dataset from `/data`
- Target + split early

## 1.a. Imports

In [1]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import re

from sentence_transformers import SentenceTransformer
from sentence_transformers import util

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer

warnings.filterwarnings("ignore")

# setting the seed
RANDOM_STATE = 871
TEST_SIZE = 0.2
DATA_DIR = Path("../data")
TARGET_COL = "toxic"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

## 1.b. Load Data

In [2]:
# ============================================================
# Data Loading - Read all parquet files and concatenate
# ============================================================
import glob

parquet_files = sorted(glob.glob('../data/*.parquet'))
print(f"Found {len(parquet_files)} parquet file(s): {parquet_files}")

data_frames = [pd.read_parquet(file, engine='fastparquet') for file in parquet_files]
maybe_df = pd.concat(data_frames, ignore_index=True) if data_frames else None

if maybe_df is None:
    print("/data/*.parquet missing")
    exit()

df: pd.DataFrame = maybe_df

print("Shape:", df.shape)
df.head()

Found 10 parquet file(s): ['../data/dataset_part_1.parquet', '../data/dataset_part_10.parquet', '../data/dataset_part_2.parquet', '../data/dataset_part_3.parquet', '../data/dataset_part_4.parquet', '../data/dataset_part_5.parquet', '../data/dataset_part_6.parquet', '../data/dataset_part_7.parquet', '../data/dataset_part_8.parquet', '../data/dataset_part_9.parquet']
Shape: (223549, 8)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [3]:
# ============================================================
# Load bad words dictionary for later analysis
# ============================================================
with open('../data/bad-words.txt') as file:
    bad_words = set(line.strip() for line in file.readlines())

print(f"Loaded {len(bad_words)} bad words from dictionary.")

Loaded 1383 bad words from dictionary.


In [4]:
# ============================================================
# Load a pretrained Sentence Transformer model
# ============================================================
# embedding_model = SentenceTransformer("BAAI/bge-m3")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 1.c. Target + Split early

In [5]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train survival rate:", float(y_train.mean()))
print("Test survival rate:", float(y_test.mean()))

Train shape: (178839, 7) Test shape: (44710, 7)
Train survival rate: 0.09565586924552251
Test survival rate: 0.09566092596734511


---
# Phase 2: Feature engineering
Taking into account the most relevant columns from the EDA 
study, the "X" sets will be added.

Additionally, an embedding will be implemented to convert 
the texts into numeric columns, and these n-embeds will 
be added as columns.

- Feature engineering functions
- Feature selection, prepare the data frame with the required columns:
    - Embeddings (numeric)
    - Character Count (numeric)
    - Word Count (numeric)
    - Uppercase Ratio (numeric)
    - Bad Words Count (numeric)
    - Unique Word Ratio (numeric)
- Preprocessing Pipeline Architecture

## 2.a. Feature Engineering

In [6]:
def extract_stripped_text(text: str) -> str:
    return re.sub(r'\W+', ' ', text.strip())

def extract_embedding_column(texts: pd.Series) -> pd.Series:
    # Backward-compatible helper: Series[object] with one ndarray per row.
    emb_df = extract_embedding_column_2(texts)
    emb_matrix = emb_df.to_numpy(dtype=np.float32, copy=False)
    return pd.Series(emb_matrix.tolist(), index=texts.index, dtype="object")

def extract_embedding_column_2(
    texts: pd.Series,
    batch_size: int = 64,
    dtype: np.dtype = np.float32,
    normalize_embeddings: bool = False,
    show_progress_bar: bool = False,
) -> pd.DataFrame:
    """
    Fast embedding extraction that returns expanded numeric columns (emb_0..emb_n).
    Optimizations: vectorized text cleanup and deduplication.
    """
    clean_texts = texts.fillna("").astype(str)
    stripped_texts = clean_texts.str.strip().str.replace(r"\W+", " ", regex=True)

    emb_dim = embedding_model.get_sentence_embedding_dimension()
    if stripped_texts.empty:
        return pd.DataFrame(index=texts.index, columns=[f"embedding_{i}" for i in range(emb_dim)], dtype=dtype)

    # Encode only unique cleaned texts, then map back to all rows.
    codes, uniques = pd.factorize(stripped_texts, sort=False)
    unique_emb = embedding_model.encode(
        uniques.tolist(),
        # batch_size=batch_size,
        # convert_to_numpy=True,
        normalize_embeddings=normalize_embeddings,
        show_progress_bar=show_progress_bar,
    ).astype(dtype, copy=False)

    emb_matrix = unique_emb[codes]
    return pd.DataFrame(emb_matrix, index=texts.index).add_prefix("embedding_")

def extract_character_count(text: str) -> int:
    if not isinstance(text, str) or not text:
        return 0

    stripped_text = extract_stripped_text(text)
    return len(stripped_text)


def extract_word_count(text: str) -> int:
    if not isinstance(text, str) or not text:
        return 0
    
    stripped_text = extract_stripped_text(text)
    words = stripped_text.split()
    return len(words)


def extract_uppercase_ratio(text: str) -> float:
    if not isinstance(text, str) or not text:
        return 0.0
    
    stripped_text = extract_stripped_text(text)
    return sum(1 for character in stripped_text if character.isalpha()) / max(len(stripped_text), 1)


def extract_unique_word_ratio(text: str) -> float:
    if not isinstance(text, str) or not text:
        return 0.0
    
    stripped_text = extract_stripped_text(text)
    words = stripped_text.lower().split()
    return len(set(words)) / max(len(words), 1)


def extract_bad_word_count(text: str) -> int:
    if not isinstance(text, str) or not text:
        return 0
    
    stripped_text = extract_stripped_text(text)
    words = re.findall(r"[a-zA-Z']+", stripped_text.lower())
    return sum(1 for word in words if word in bad_words)


def add_engineered_features(X_in: pd.DataFrame) -> pd.DataFrame:
    """
    Create engineered features:
    - title from Name
    - deck from Cabin
    - has_cabin flag
    - family_size and is_alone
    """
    X = X_in.copy()

    emb_df = extract_embedding_column_2(X["comment_text"], batch_size=2048, dtype=np.float32)
    X = pd.concat([X, emb_df], axis=1)

    X["character_count"] = X["comment_text"].apply(extract_character_count)
    X["word_count"] = X["comment_text"].apply(extract_word_count)
    X["uppercase_ratio"] = X["comment_text"].apply(extract_uppercase_ratio)
    X["unique_word_ratio"] = X["comment_text"].apply(extract_unique_word_ratio)
    X["bad_word_count"] = X["comment_text"].apply(extract_bad_word_count)

    return X

**Applying the functions**

In [7]:
X_train_fe = add_engineered_features(X_train)
X_train_fe.head()

,id,comment_text,severe_toxic,obscene,threat,insult,identity_hate,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,embedding_6,embedding_7,embedding_8,embedding_9,embedding_10,embedding_11,embedding_12,embedding_13,embedding_14,embedding_15,embedding_16,embedding_17,embedding_18,embedding_19,embedding_20,embedding_21,embedding_22,embedding_23,embedding_24,embedding_25,embedding_26,embedding_27,embedding_28,embedding_29,embedding_30,embedding_31,embedding_32,embedding_33,embedding_34,embedding_35,embedding_36,embedding_37,embedding_38,embedding_39,embedding_40,embedding_41,embedding_42,embedding_43,embedding_44,embedding_45,embedding_46,embedding_47,embedding_48,embedding_49,embedding_50,embedding_51,embedding_52,embedding_53,embedding_54,embedding_55,embedding_56,embedding_57,embedding_58,embedding_59,embedding_60,embedding_61,embedding_62,embedding_63,embedding_64,embedding_65,embedding_66,embedding_67,embedding_68,embedding_69,embedding_70,embedding_71,embedding_72,embedding_73,embedding_74,embedding_75,embedding_76,embedding_77,embedding_78,embedding_79,embedding_80,embedding_81,embedding_82,embedding_83,embedding_84,embedding_85,embedding_86,embedding_87,embedding_88,embedding_89,embedding_90,embedding_91,embedding_92,...,embedding_289,embedding_290,embedding_291,embedding_292,embedding_293,embedding_294,embedding_295,embedding_296,embedding_297,embedding_298,embedding_299,embedding_300,embedding_301,embedding_302,embedding_303,embedding_304,embedding_305,embedding_306,embedding_307,embedding_308,embedding_309,embedding_310,embedding_311,embedding_312,embedding_313,embedding_314,embedding_315,embedding_316,embedding_317,embedding_318,embedding_319,embedding_320,embedding_321,embedding_322,embedding_323,embedding_324,embedding_325,embedding_326,embedding_327,embedding_328,embedding_329,embedding_330,embedding_331,embedding_332,embedding_333,embedding_334,embedding_335,embedding_336,embedding_337,embedding_338,embedding_339,embedding_340,embedding_341,embedding_342,embedding_343,embedding_344,embedding_345,embedding_346,embedding_347,embedding_348,embedding_349,embedding_350,embedding_351,embedding_352,embedding_353,embedding_354,embedding_355,embedding_356,embedding_357,embedding_358,embedding_359,embedding_360,embedding_361,embedding_362,embedding_363,embedding_364,embedding_365,embedding_366,embedding_367,embedding_368,embedding_369,embedding_370,embedding_371,embedding_372,embedding_373,embedding_374,embedding_375,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383,character_count,word_count,uppercase_ratio,unique_word_ratio,bad_word_count
182602,02b2940d87dae670,israeli names for Santa Claus-I have been aske...,0,0,0,0,0,0.010778,0.116142,-0.050181,0.017789,-0.038281,0.032517,0.046764,-0.037285,0.038807,0.004361,-0.015046,-0.039248,-0.010371,0.064799,-0.025403,-0.003864,-0.084063,0.065340,0.034263,-0.060697,0.029021,-0.007443,0.026138,-0.094956,0.055924,0.023140,-0.037283,-0.000860,0.022800,-0.017720,-0.051050,0.020100,0.114541,-0.047197,0.033338,0.041709,-0.006318,0.095160,0.002689,0.013510,0.021818,0.040743,-0.011047,-0.065431,-0.062714,0.081632,-0.041139,0.031355,-0.025265,0.076007,-0.096970,-0.050240,0.082068,0.036048,-0.030377,-0.079856,0.010736,-0.064586,-0.031255,-0.017916,-0.027553,0.069962,-0.027604,-0.024341,-0.034575,0.016020,0.046405,-0.036407,-0.075552,-0.001715,0.022069,0.066852,0.040781,-0.002498,-0.067456,-0.080119,0.011274,-0.029626,-0.054810,0.022828,-0.036547,-0.033457,-0.050944,0.005272,-0.034081,0.087928,-0.054613,0.003423,-0.007431,-0.017565,-0.021701,-0.106365,0.130663,...,-0.126142,-0.018015,-0.018906,0.089326,-0.054300,0.072538,0.021262,0.031782,-0.018192,-0.140839,0.034677,0.053196,0.025451,-0.001475,-0.027822,0.030317,0.027799,-0.055500,0.140844,0.030325,0.010080,0.037639,0.036834,0.003874,0.062687,0.027449,0.013838,0.009156,-0.000575,-0.009012,-2.294656e-08,0.110499,-0.016756,-0.051544,-0.012859,-0.008973,-0.03

In [8]:
X_test_fe = add_engineered_features(X_test)
X_test_fe.head()

,id,comment_text,severe_toxic,obscene,threat,insult,identity_hate,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,embedding_6,embedding_7,embedding_8,embedding_9,embedding_10,embedding_11,embedding_12,embedding_13,embedding_14,embedding_15,embedding_16,embedding_17,embedding_18,embedding_19,embedding_20,embedding_21,embedding_22,embedding_23,embedding_24,embedding_25,embedding_26,embedding_27,embedding_28,embedding_29,embedding_30,embedding_31,embedding_32,embedding_33,embedding_34,embedding_35,embedding_36,embedding_37,embedding_38,embedding_39,embedding_40,embedding_41,embedding_42,embedding_43,embedding_44,embedding_45,embedding_46,embedding_47,embedding_48,embedding_49,embedding_50,embedding_51,embedding_52,embedding_53,embedding_54,embedding_55,embedding_56,embedding_57,embedding_58,embedding_59,embedding_60,embedding_61,embedding_62,embedding_63,embedding_64,embedding_65,embedding_66,embedding_67,embedding_68,embedding_69,embedding_70,embedding_71,embedding_72,embedding_73,embedding_74,embedding_75,embedding_76,embedding_77,embedding_78,embedding_79,embedding_80,embedding_81,embedding_82,embedding_83,embedding_84,embedding_85,embedding_86,embedding_87,embedding_88,embedding_89,embedding_90,embedding_91,embedding_92,...,embedding_289,embedding_290,embedding_291,embedding_292,embedding_293,embedding_294,embedding_295,embedding_296,embedding_297,embedding_298,embedding_299,embedding_300,embedding_301,embedding_302,embedding_303,embedding_304,embedding_305,embedding_306,embedding_307,embedding_308,embedding_309,embedding_310,embedding_311,embedding_312,embedding_313,embedding_314,embedding_315,embedding_316,embedding_317,embedding_318,embedding_319,embedding_320,embedding_321,embedding_322,embedding_323,embedding_324,embedding_325,embedding_326,embedding_327,embedding_328,embedding_329,embedding_330,embedding_331,embedding_332,embedding_333,embedding_334,embedding_335,embedding_336,embedding_337,embedding_338,embedding_339,embedding_340,embedding_341,embedding_342,embedding_343,embedding_344,embedding_345,embedding_346,embedding_347,embedding_348,embedding_349,embedding_350,embedding_351,embedding_352,embedding_353,embedding_354,embedding_355,embedding_356,embedding_357,embedding_358,embedding_359,embedding_360,embedding_361,embedding_362,embedding_363,embedding_364,embedding_365,embedding_366,embedding_367,embedding_368,embedding_369,embedding_370,embedding_371,embedding_372,embedding_373,embedding_374,embedding_375,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383,character_count,word_count,uppercase_ratio,unique_word_ratio,bad_word_count
108295,e5e15c2f432a53fd,"I should stay out of this altogether, probably...",0,0,0,0,0,0.015799,-0.041383,0.023815,0.006023,0.072239,0.060871,-0.013208,-0.018158,0.085554,-0.033026,-0.045481,0.064122,0.048828,-0.088602,-0.081924,0.115147,-0.027516,-0.043606,-0.055980,0.052723,-0.105649,0.012360,0.060160,0.117017,-0.038671,-0.109735,-0.061358,-0.037237,-0.081780,-0.051702,-0.109844,0.018792,-0.024508,-0.035221,-0.034390,0.071426,0.070065,0.034991,0.038952,-0.089696,0.018194,-0.087624,-0.015175,-0.109424,-0.013958,0.034954,0.002559,-0.011931,-0.043984,-0.064842,-0.059949,-0.039892,0.020066,-0.059996,-0.077292,0.059512,-0.019451,0.060104,0.053265,-0.068815,0.042752,0.008117,0.009561,0.009596,0.001944,-0.001170,-0.059755,0.038667,-0.028236,-0.001351,-0.016528,0.041599,-0.005443,0.071761,-0.022266,-0.016608,0.067201,0.002324,0.032988,0.047587,0.042961,-0.008830,0.064403,-0.031646,0.000399,-0.038011,-0.003133,0.024162,0.004171,-0.027877,0.003454,-0.027608,0.117014,...,-0.013786,-0.043303,0.069160,0.018982,0.023660,-0.018352,-0.042803,0.002411,-0.024371,0.052675,0.050221,0.046910,-0.067706,0.059614,-0.008903,0.022847,-0.001151,0.021255,0.033106,0.068380,-0.050190,0.017481,0.057622,0.008229,0.052370,-0.009124,0.029751,-0.019356,0.023890,0.046748,-5.576285e-08,0.096646,-0.047683,0.063403,0.052332,0.053243,0.031413

## 2.b. Feature Selection
The idea is to remove irrelevant columns from the dataframe; taking into consideration the EDA studies previously reviewed.
The comment column is then removed as we no longer need it (it was computationally replaced by the embeddings).

In [9]:
drop_columns = [
    "id",           # identifier
    "comment_text", # turned into a vector (embeddings)
    "severe_toxic", # substituted by toxic, and it is a "Y" set value
    "obscene",      # substituted by toxic, and it is a "Y" set value
    "threat",       # substituted by toxic, and it is a "Y" set value
    "insult",       # substituted by toxic, and it is a "Y" set value
    "identity_hate" # substituted by toxic, and it is a "Y" set value
]

X_train_fe = X_train_fe.drop(columns=drop_columns, errors="ignore")
X_test_fe = X_test_fe.drop(columns=drop_columns, errors="ignore")

print("After FE + drop:", X_train_fe.shape, X_test_fe.shape)

X_train_fe.head()

After FE + drop: (178839, 389) (44710, 389)


,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,embedding_6,embedding_7,embedding_8,embedding_9,embedding_10,embedding_11,embedding_12,embedding_13,embedding_14,embedding_15,embedding_16,embedding_17,embedding_18,embedding_19,embedding_20,embedding_21,embedding_22,embedding_23,embedding_24,embedding_25,embedding_26,embedding_27,embedding_28,embedding_29,embedding_30,embedding_31,embedding_32,embedding_33,embedding_34,embedding_35,embedding_36,embedding_37,embedding_38,embedding_39,embedding_40,embedding_41,embedding_42,embedding_43,embedding_44,embedding_45,embedding_46,embedding_47,embedding_48,embedding_49,embedding_50,embedding_51,embedding_52,embedding_53,embedding_54,embedding_55,embedding_56,embedding_57,embedding_58,embedding_59,embedding_60,embedding_61,embedding_62,embedding_63,embedding_64,embedding_65,embedding_66,embedding_67,embedding_68,embedding_69,embedding_70,embedding_71,embedding_72,embedding_73,embedding_74,embedding_75,embedding_76,embedding_77,embedding_78,embedding_79,embedding_80,embedding_81,embedding_82,embedding_83,embedding_84,embedding_85,embedding_86,embedding_87,embedding_88,embedding_89,embedding_90,embedding_91,embedding_92,embedding_93,embedding_94,embedding_95,embedding_96,embedding_97,embedding_98,embedding_99,...,embedding_289,embedding_290,embedding_291,embedding_292,embedding_293,embedding_294,embedding_295,embedding_296,embedding_297,embedding_298,embedding_299,embedding_300,embedding_301,embedding_302,embedding_303,embedding_304,embedding_305,embedding_306,embedding_307,embedding_308,embedding_309,embedding_310,embedding_311,embedding_312,embedding_313,embedding_314,embedding_315,embedding_316,embedding_317,embedding_318,embedding_319,embedding_320,embedding_321,embedding_322,embedding_323,embedding_324,embedding_325,embedding_326,embedding_327,embedding_328,embedding_329,embedding_330,embedding_331,embedding_332,embedding_333,embedding_334,embedding_335,embedding_336,embedding_337,embedding_338,embedding_339,embedding_340,embedding_341,embedding_342,embedding_343,embedding_344,embedding_345,embedding_346,embedding_347,embedding_348,embedding_349,embedding_350,embedding_351,embedding_352,embedding_353,embedding_354,embedding_355,embedding_356,embedding_357,embedding_358,embedding_359,embedding_360,embedding_361,embedding_362,embedding_363,embedding_364,embedding_365,embedding_366,embedding_367,embedding_368,embedding_369,embedding_370,embedding_371,embedding_372,embedding_373,embedding_374,embedding_375,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383,character_count,word_count,uppercase_ratio,unique_word_ratio,bad_word_count
182602,0.010778,0.116142,-0.050181,0.017789,-0.038281,0.032517,0.046764,-0.037285,0.038807,0.004361,-0.015046,-0.039248,-0.010371,0.064799,-0.025403,-0.003864,-0.084063,0.065340,0.034263,-0.060697,0.029021,-0.007443,0.026138,-0.094956,0.055924,0.023140,-0.037283,-0.000860,0.022800,-0.017720,-0.051050,0.020100,0.114541,-0.047197,0.033338,0.041709,-0.006318,0.095160,0.002689,0.013510,0.021818,0.040743,-0.011047,-0.065431,-0.062714,0.081632,-0.041139,0.031355,-0.025265,0.076007,-0.096970,-0.050240,0.082068,0.036048,-0.030377,-0.079856,0.010736,-0.064586,-0.031255,-0.017916,-0.027553,0.069962,-0.027604,-0.024341,-0.034575,0.016020,0.046405,-0.036407,-0.075552,-0.001715,0.022069,0.066852,0.040781,-0.002498,-0.067456,-0.080119,0.011274,-0.029626,-0.054810,0.022828,-0.036547,-0.033457,-0.050944,0.005272,-0.034081,0.087928,-0.054613,0.003423,-0.007431,-0.017565,-0.021701,-0.106365,0.130663,0.006929,0.072951,-0.000720,0.026877,0.046059,-0.024033,-0.047640,...,-0.126142,-0.018015,-0.018906,0.089326,-0.054300,0.072538,0.021262,0.031782,-0.018192,-0.140839,0.034677,0.053196,0.025451,-0.001475,-0.027822,0.030317,0.027799,-0.055500,0.140844,0.030325,0.010080,0.037639,0.036834,0.003874,0.062687,0.027449,0.013838,0.009156,-0.000575,-0.009012,-2.294656e-08,0.110499,-0.016756,-0.051544,-0.012859,

In [10]:
numeric_features = ["character_count", "word_count", "uppercase_ratio", "unique_word_ratio", "bad_word_count"] + [f"embedding_{i}" for i in range(embedding_model.get_sentence_embedding_dimension())]

# Ensure only existing columns are used (robustness)
def keep_existing(cols: list[str], df_: pd.DataFrame) -> list[str]:
    return [c for c in cols if c in df_.columns]

numeric_features = keep_existing(numeric_features, X_train_fe)

numeric_features

['character_count',
 'word_count',
 'uppercase_ratio',
 'unique_word_ratio',
 'bad_word_count',
 'embedding_0',
 'embedding_1',
 'embedding_2',
 'embedding_3',
 'embedding_4',
 'embedding_5',
 'embedding_6',
 'embedding_7',
 'embedding_8',
 'embedding_9',
 'embedding_10',
 'embedding_11',
 'embedding_12',
 'embedding_13',
 'embedding_14',
 'embedding_15',
 'embedding_16',
 'embedding_17',
 'embedding_18',
 'embedding_19',
 'embedding_20',
 'embedding_21',
 'embedding_22',
 'embedding_23',
 'embedding_24',
 'embedding_25',
 'embedding_26',
 'embedding_27',
 'embedding_28',
 'embedding_29',
 'embedding_30',
 'embedding_31',
 'embedding_32',
 'embedding_33',
 'embedding_34',
 'embedding_35',
 'embedding_36',
 'embedding_37',
 'embedding_38',
 'embedding_39',
 'embedding_40',
 'embedding_41',
 'embedding_42',
 'embedding_43',
 'embedding_44',
 'embedding_45',
 'embedding_46',
 'embedding_47',
 'embedding_48',
 'embedding_49',
 'embedding_50',
 'embedding_51',
 'embedding_52',
 'embedding_5

### Some registers observations
The model being used for embedding has a peculiarity: when calculating the dimensions (of which there are 384), the algorithm truncates after 256 words. Therefore, some records (see analysis) will experience this issue. However, this is mitigated by the other columns, which add more information (and therefore value) to the record in this embedding analysis.


## 2.c. Preprocessing Pipeline Architecture
This involves saving the data structure to prevent data loss. Note that because there is a large amount of data, generating the embeds takes a significant amount of time, making this step crucial.

In [11]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

preprocess_pipeline = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
    ],
    remainder="drop",
)

In [ ]:
X_train_fe.head()

,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,embedding_6,embedding_7,embedding_8,embedding_9,embedding_10,embedding_11,embedding_12,embedding_13,embedding_14,embedding_15,embedding_16,embedding_17,embedding_18,embedding_19,embedding_20,embedding_21,embedding_22,embedding_23,embedding_24,embedding_25,embedding_26,embedding_27,embedding_28,embedding_29,embedding_30,embedding_31,embedding_32,embedding_33,embedding_34,embedding_35,embedding_36,embedding_37,embedding_38,embedding_39,embedding_40,embedding_41,embedding_42,embedding_43,embedding_44,embedding_45,embedding_46,embedding_47,embedding_48,embedding_49,embedding_50,embedding_51,embedding_52,embedding_53,embedding_54,embedding_55,embedding_56,embedding_57,embedding_58,embedding_59,embedding_60,embedding_61,embedding_62,embedding_63,embedding_64,embedding_65,embedding_66,embedding_67,embedding_68,embedding_69,embedding_70,embedding_71,embedding_72,embedding_73,embedding_74,embedding_75,embedding_76,embedding_77,embedding_78,embedding_79,embedding_80,embedding_81,embedding_82,embedding_83,embedding_84,embedding_85,embedding_86,embedding_87,embedding_88,embedding_89,embedding_90,embedding_91,embedding_92,embedding_93,embedding_94,embedding_95,embedding_96,embedding_97,embedding_98,embedding_99,...,embedding_289,embedding_290,embedding_291,embedding_292,embedding_293,embedding_294,embedding_295,embedding_296,embedding_297,embedding_298,embedding_299,embedding_300,embedding_301,embedding_302,embedding_303,embedding_304,embedding_305,embedding_306,embedding_307,embedding_308,embedding_309,embedding_310,embedding_311,embedding_312,embedding_313,embedding_314,embedding_315,embedding_316,embedding_317,embedding_318,embedding_319,embedding_320,embedding_321,embedding_322,embedding_323,embedding_324,embedding_325,embedding_326,embedding_327,embedding_328,embedding_329,embedding_330,embedding_331,embedding_332,embedding_333,embedding_334,embedding_335,embedding_336,embedding_337,embedding_338,embedding_339,embedding_340,embedding_341,embedding_342,embedding_343,embedding_344,embedding_345,embedding_346,embedding_347,embedding_348,embedding_349,embedding_350,embedding_351,embedding_352,embedding_353,embedding_354,embedding_355,embedding_356,embedding_357,embedding_358,embedding_359,embedding_360,embedding_361,embedding_362,embedding_363,embedding_364,embedding_365,embedding_366,embedding_367,embedding_368,embedding_369,embedding_370,embedding_371,embedding_372,embedding_373,embedding_374,embedding_375,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383,character_count,word_count,uppercase_ratio,unique_word_ratio,bad_word_count
182602,0.010778,0.116142,-0.050181,0.017789,-0.038281,0.032517,0.046764,-0.037285,0.038807,0.004361,-0.015046,-0.039248,-0.010371,0.064799,-0.025403,-0.003864,-0.084063,0.065340,0.034263,-0.060697,0.029021,-0.007443,0.026138,-0.094956,0.055924,0.023140,-0.037283,-0.000860,0.022800,-0.017720,-0.051050,0.020100,0.114541,-0.047197,0.033338,0.041709,-0.006318,0.095160,0.002689,0.013510,0.021818,0.040743,-0.011047,-0.065431,-0.062714,0.081632,-0.041139,0.031355,-0.025265,0.076007,-0.096970,-0.050240,0.082068,0.036048,-0.030377,-0.079856,0.010736,-0.064586,-0.031255,-0.017916,-0.027553,0.069962,-0.027604,-0.024341,-0.034575,0.016020,0.046405,-0.036407,-0.075552,-0.001715,0.022069,0.066852,0.040781,-0.002498,-0.067456,-0.080119,0.011274,-0.029626,-0.054810,0.022828,-0.036547,-0.033457,-0.050944,0.005272,-0.034081,0.087928,-0.054613,0.003423,-0.007431,-0.017565,-0.021701,-0.106365,0.130663,0.006929,0.072951,-0.000720,0.026877,0.046059,-0.024033,-0.047640,...,-0.126142,-0.018015,-0.018906,0.089326,-0.054300,0.072538,0.021262,0.031782,-0.018192,-0.140839,0.034677,0.053196,0.025451,-0.001475,-0.027822,0.030317,0.027799,-0.055500,0.140844,0.030325,0.010080,0.037639,0.036834,0.003874,0.062687,0.027449,0.013838,0.009156,-0.000575,-0.009012,-2.294656e-08,0.110499,-0.016756,-0.051544,-0.012859,